# Lectra AI — GPU tunnel worker

Runs the project's real audio pipeline (`LectraAIPipeline`, unmodified) on Kaggle's free GPU, and exposes it to your local backend over a public tunnel URL.

**Before running:** Notebook Settings (right sidebar) → Accelerator → **GPU T4 x2** (or P100) → Internet → **On**.

**Also before running:** Add-ons → Secrets → attach two secrets to this notebook:
- `GPU_TUNNEL_TOKEN` — any long random string you make up. This is the shared password your local backend uses to talk to this worker; anyone with the tunnel URL *and* this token can submit jobs.
- `HF_TOKEN` — your HuggingFace token (needed for speaker diarization), same one from your local `.env`.

Run every cell top to bottom, then copy the printed tunnel URL (and the token you chose) into your local `.env` as `GPU_TUNNEL_URL` / `GPU_TUNNEL_TOKEN`, and restart your local backend. See `gpu_tunnel/README.md` for the full walkthrough.

Session limits (Kaggle free tier, subject to change): ~9-12h per session, ~30 GPU-hours/week. The tunnel URL changes every time you restart this notebook — just paste the new one in.

In [ ]:
# --- 1. Confirm we actually have a GPU before doing anything else ---
import torch

assert torch.cuda.is_available(), (
    "No GPU detected! Notebook Settings (right sidebar) -> Accelerator -> "
    "GPU T4 x2 (or P100). Also check Settings -> Internet -> On."
)
print(f"GPU OK: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- 2. Get the actual pipeline code (public repo, no auth needed) ---
# cd to a stable parent FIRST, then rm -rf + clone - never delete a
# directory that is an ancestor of the shell's current cwd (that leaves
# the shell broken on re-runs within the same session).
%cd /kaggle/working
!rm -rf repo && git clone --depth 1 https://github.com/Hanzala-12/lectra-ai.git repo
%cd /kaggle/working/repo

In [ ]:
# --- 3. Install ONLY what is missing from Kaggle's own image ---
# kaggle_requirements.txt deliberately excludes torch/torchaudio/numpy/scipy
# so pip's resolver leaves Kaggle's preinstalled CUDA build alone.
#
# Two things this needs that are easy to get wrong (found the hard way):
# 1. Always install via {sys.executable} -m pip, never bare "pip install" -
#    on Kaggle the plain pip on PATH can resolve to a DIFFERENT Python than
#    the one this notebook's kernel actually runs (confirmed: bare
#    "pip install" reported success while the kernel's own interpreter
#    still couldn't import any of it).
# 2. deepfilternet depends on DeepFilterLib, a Rust extension with no
#    prebuilt wheel for Kaggle's Python version - it needs an actual Rust
#    toolchain to build from source, which Kaggle doesn't ship by default.
import sys, os, subprocess

rust_install = subprocess.run(
    ["bash", "-c",
     "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
     "| sh -s -- -y --default-toolchain stable"],
    capture_output=True, text=True,
)
assert rust_install.returncode == 0, rust_install.stderr[-2000:]
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
print("Rust toolchain installed.")

install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "gpu_tunnel/kaggle_requirements.txt"],
    capture_output=True, text=True, env=os.environ,
)
if install.returncode != 0:
    print("First install attempt failed, retrying once...")
    install = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", "gpu_tunnel/kaggle_requirements.txt"],
        capture_output=True, text=True, env=os.environ,
    )
print(install.stdout[-3000:])
assert install.returncode == 0, install.stderr[-2000:]

import torch
assert torch.cuda.is_available(), (
    "GPU was available before installing requirements but NOT after - "
    "something in kaggle_requirements.txt pulled in a CPU-only torch build. "
    "Do not add torch/torchaudio/numpy/scipy to that file."
)
print(f"Still GPU OK after installs: {torch.cuda.get_device_name(0)}")

In [ ]:
# --- 4. Load secrets (see the Add-ons -> Secrets setup note at the top) ---
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

try:
    os.environ["GPU_TUNNEL_TOKEN"] = secrets.get_secret("GPU_TUNNEL_TOKEN")
except Exception:
    raise RuntimeError(
        "Couldn't read the GPU_TUNNEL_TOKEN secret. Add-ons -> Secrets -> "
        "add one named exactly GPU_TUNNEL_TOKEN (any random string you make "
        "up) -> make sure it's attached/enabled for THIS notebook."
    )

try:
    os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
except Exception:
    print(
        "No HF_TOKEN secret found - diarization will fall back to VAD "
        "(one undifferentiated 'speaker'), same as running locally without "
        "it. Add one via Add-ons -> Secrets if you want real diarization."
    )

print("Secrets loaded.")

In [ ]:
# --- 5. Start the worker API in the background ---
import threading
import time
import asyncio
import uvicorn
import uvicorn.config

from gpu_tunnel.worker_app import app

# Kaggle's preinstalled uvicorn (confirmed: 0.46.0, from
# /usr/local/lib/python3.12/dist-packages) ships an internally-inconsistent
# build: Config.get_loop_factory() (the method uvicorn.run() actually calls
# to pick an event loop, replacing the older setup_event_loop() as of
# upstream uvicorn 0.36.0) expects each uvicorn/loops/<name>.py submodule to
# expose a `<name>_loop_factory` callable - but the loops/ submodules
# actually installed on this image are stale pre-0.36.0 files that only
# define the old `asyncio_setup`/etc names. Confirmed live: this makes
# uvicorn.run() fail with `uvicorn.importer.ImportFromStringError: Attribute
# "asyncio_loop_factory" not found in module "uvicorn.loops.asyncio"`
# regardless of the `loop=` kwarg ("auto" and "asyncio" both hit it, since
# both resolve through the same broken lookup). Not something fixable via
# kaggle_requirements.txt (pinning a different uvicorn version doesn't
# change what Kaggle's base image has already put on disk) - patching
# Config.get_loop_factory() to just return a plain stdlib event-loop
# factory sidesteps the broken lookup entirely. asyncio.new_event_loop is
# itself already a zero-arg callable returning a fresh loop, i.e. exactly
# what Python 3.12's asyncio.Runner(loop_factory=...) contract expects, so
# no wrapper function is needed. Same "monkeypatch before the real call"
# pattern already used for torchaudio (see deepfilter_processor.py).
uvicorn.config.Config.get_loop_factory = lambda self: asyncio.new_event_loop


def _run_server():
    uvicorn.run(app, host="0.0.0.0", port=8800, log_level="info")


threading.Thread(target=_run_server, daemon=True).start()
time.sleep(5)
print("Worker API starting on port 8800.")

In [ ]:
# --- 6. Open the tunnel (cloudflared quick tunnel - no account needed) ---
import re
import subprocess

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8800"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

print("Waiting for cloudflared to establish the tunnel...")
url = None
for line in proc.stdout:
    if "trycloudflare.com" in line:
        m = re.search(r"https://[a-zA-Z0-9\-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0)
            break

print("\n" + "=" * 70)
print(f"TUNNEL URL:  {url}")
print("=" * 70)
print("Paste into your local .env:")
print(f"  GPU_TUNNEL_URL={url}")
print("  GPU_TUNNEL_TOKEN=<the same value you put in the GPU_TUNNEL_TOKEN secret above>")
print("Then restart your local backend (python backend.py).")

In [ ]:
# --- 7. Keep this cell running to keep the worker + tunnel alive ---
# Stop this cell (or close the notebook) to shut the worker down - your
# local backend will automatically fall back to local CPU processing.
import time

print("Worker is live. Keep this cell running - stopping it ends the tunnel.")
while True:
    time.sleep(300)
    print(f"[{time.strftime('%H:%M:%S')}] still running...")